In [1]:
list.of.packages <- c("tidyverse","data.table","stargazer")
new.packages <- list.of.packages[!(list.of.packages %in% installed.packages()[,"Package"])]
if(length(new.packages)) install.packages(new.packages, repos = "http://cran.us.r-project.org")

invisible(lapply(list.of.packages, library, character.only = TRUE))

options(repr.matrix.max.rows=500, repr.matrix.max.cols=1200, message=FALSE, warning=TRUE)  

── Attaching core tidyverse packages ────────────────────────────────────────────────────────────────────────────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.0     ✔ tibble    3.3.0
✔ lubridate 1.9.4     ✔ tidyr     1.3.1
✔ purrr     1.2.0     
── Conflicts ──────────────────────────────────────────────────────────────────────────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors

Attachement du package : ‘data.table’


Les objets suivants sont masqués depuis ‘package:lubridate’:

    hour, isoweek, mday, minute, month, quarter, second, wday, week,
    yday, year


Les objets suivants sont masqués depuis ‘package:dplyr’:

    between, first, last


L'objet suivant est masqué depuis ‘package:purrr’:

    transp

# Longer dataframe

In [2]:
df_deg <- fread("./interm/df_WB_cells_CDC_longer.csv")
sprintf("%i x %i dataframe", nrow(df_deg), ncol(df_deg))
head(df_deg,2)

[1] "30298932 x 7 dataframe"

Measured_date_CDC,cell_x,cell_y,Temp_bin,N_hours,Temp,Day_Min
<int>,<dbl>,<dbl>,<chr>,<int>,<int>,<int>
42474,91.75,27.50,T0DayMin1,1,0,1
42660,96.75,28.25,T0DayMin1,4,0,1


# Categorize T per hour into bins

In [3]:
data.frame(df_deg) %>% 
        select(Temp)%>%
        stargazer(nobs=TRUE, digits = 2,
                  summary.stat = c("n","mean", "sd","min","p25","median","p75","max"),
                  title = c("Summary statistics"),
                  type="text"
                     )


Summary statistics
Statistic     N      Mean  St. Dev. Min Pctl(25) Median Pctl(75) Max
--------------------------------------------------------------------
Temp      30,298,932 19.78   7.13   -28    15      21      25    37 
--------------------------------------------------------------------


In [4]:
thresholds_WB <- c(-28,14,16,18,20,22,24,26,28,37)

In [5]:
df_deg_bins <- df_deg %>%
                  mutate(WB_bin = cut(Temp, breaks = thresholds_WB, 
                                     labels = c("WB_Bin_inf14","WB_Bin_14_16","WB_Bin_16_18",
                                                "WB_Bin_18_20","WB_Bin_20_22","WB_Bin_22_24",
                                                "WB_Bin_24_26","WB_Bin_26_28","WB_Bin_sup28"),
                                     include.lowest = TRUE))
sprintf("%i x %i dataframe", nrow(df_deg_bins), ncol(df_deg_bins))
head(df_deg_bins,2)

[1] "30298932 x 8 dataframe"

Measured_date_CDC,cell_x,cell_y,Temp_bin,N_hours,Temp,Day_Min,WB_bin
<int>,<dbl>,<dbl>,<chr>,<int>,<int>,<int>,<fct>
42474,91.75,27.50,T0DayMin1,1,0,1,WB_Bin_inf14
42660,96.75,28.25,T0DayMin1,4,0,1,WB_Bin_inf14


# Pivot wider

In [6]:
df_deg_wide <- df_deg_bins %>%
                        select(Measured_date_CDC,cell_x,cell_y,Day_Min,WB_bin,N_hours)%>%
                        group_by(Measured_date_CDC,cell_x,cell_y,Day_Min,WB_bin)%>%
                        summarize(N_hours=sum(N_hours))%>%
                        ungroup()%>%
                        pivot_wider(names_from = WB_bin, values_from = N_hours)%>%
                        ungroup()%>%
                        mutate_all(~replace_na(., 0))
sprintf("%i x %i dataframe", nrow(df_deg_wide), ncol(df_deg_wide))
head(df_deg_wide,2)

`summarise()` has grouped output by 'Measured_date_CDC', 'cell_x', 'cell_y', 'Day_Min'. You can override using the `.groups` argument.


[1] "3801750 x 13 dataframe"

Measured_date_CDC,cell_x,cell_y,Day_Min,WB_Bin_18_20,WB_Bin_20_22,WB_Bin_22_24,WB_Bin_24_26,WB_Bin_16_18,WB_Bin_14_16,WB_Bin_inf14,WB_Bin_26_28,WB_Bin_sup28
<int>,<dbl>,<dbl>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>
42022,74,15.25,1,9,5,3,7,0,0,0,0,0
42022,74,15.25,2,8,2,3,7,4,0,0,0,0


In [7]:
df_deg_Nhours_30 <- df_deg_wide %>%
                        select(-Day_Min)%>%
                        group_by(Measured_date_CDC,cell_x,cell_y)%>%
                        summarise_all(list(Nhours_on30 =~ sum(.)))%>%
                        ungroup()
sprintf("%i x %i dataframe", nrow(df_deg_Nhours_30), ncol(df_deg_Nhours_30))
head(df_deg_Nhours_30,2)

[1] "126725 x 12 dataframe"

Measured_date_CDC,cell_x,cell_y,WB_Bin_18_20_Nhours_on30,WB_Bin_20_22_Nhours_on30,WB_Bin_22_24_Nhours_on30,WB_Bin_24_26_Nhours_on30,WB_Bin_16_18_Nhours_on30,WB_Bin_14_16_Nhours_on30,WB_Bin_inf14_Nhours_on30,WB_Bin_26_28_Nhours_on30,WB_Bin_sup28_Nhours_on30
<int>,<dbl>,<dbl>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>
42022,74,15.25,183,153,180,91,82,25,4,2,0
42022,74,15.50,124,97,140,107,118,79,48,7,0


In [8]:
df_deg_Nhours_20 <- df_deg_wide %>%
                        filter(Day_Min <= 20)%>%
                        select(-Day_Min)%>%
                        group_by(Measured_date_CDC,cell_x,cell_y)%>%
                        summarise_all(list(Nhours_on20 =~ sum(.)))%>%
                        ungroup()
sprintf("%i x %i dataframe", nrow(df_deg_Nhours_20), ncol(df_deg_Nhours_20))
head(df_deg_Nhours_20,2)

[1] "126725 x 12 dataframe"

Measured_date_CDC,cell_x,cell_y,WB_Bin_18_20_Nhours_on20,WB_Bin_20_22_Nhours_on20,WB_Bin_22_24_Nhours_on20,WB_Bin_24_26_Nhours_on20,WB_Bin_16_18_Nhours_on20,WB_Bin_14_16_Nhours_on20,WB_Bin_inf14_Nhours_on20,WB_Bin_26_28_Nhours_on20,WB_Bin_sup28_Nhours_on20
<int>,<dbl>,<dbl>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>
42022,74,15.25,110,112,119,55,54,25,4,1,0
42022,74,15.50,90,68,96,68,72,41,43,2,0


In [9]:
df_deg_Nhours_10 <- df_deg_wide %>%
                        filter(Day_Min <= 10)%>%
                        select(-Day_Min)%>%
                        group_by(Measured_date_CDC,cell_x,cell_y)%>%
                        summarise_all(list(Nhours_on10 =~ sum(.)))%>%
                        ungroup()
sprintf("%i x %i dataframe", nrow(df_deg_Nhours_10), ncol(df_deg_Nhours_10))
head(df_deg_Nhours_10,2)

[1] "126725 x 12 dataframe"

Measured_date_CDC,cell_x,cell_y,WB_Bin_18_20_Nhours_on10,WB_Bin_20_22_Nhours_on10,WB_Bin_22_24_Nhours_on10,WB_Bin_24_26_Nhours_on10,WB_Bin_16_18_Nhours_on10,WB_Bin_14_16_Nhours_on10,WB_Bin_inf14_Nhours_on10,WB_Bin_26_28_Nhours_on10,WB_Bin_sup28_Nhours_on10
<int>,<dbl>,<dbl>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>
42022,74,15.25,53,35,52,24,47,25,4,0,0
42022,74,15.50,28,27,46,24,36,36,43,0,0


# Merge

In [10]:
result <- data.frame(df_deg_Nhours_30) %>%
                    left_join(df_deg_Nhours_20, by = c("Measured_date_CDC","cell_x","cell_y"))%>%
                    left_join(df_deg_Nhours_10, by = c("Measured_date_CDC","cell_x","cell_y"))
sprintf("%i x %i dataframe", nrow(result), ncol(result))
head(result,2)

[1] "126725 x 30 dataframe"

,Measured_date_CDC,cell_x,cell_y,WB_Bin_18_20_Nhours_on30,WB_Bin_20_22_Nhours_on30,WB_Bin_22_24_Nhours_on30,WB_Bin_24_26_Nhours_on30,WB_Bin_16_18_Nhours_on30,WB_Bin_14_16_Nhours_on30,WB_Bin_inf14_Nhours_on30,WB_Bin_26_28_Nhours_on30,WB_Bin_sup28_Nhours_on30,WB_Bin_18_20_Nhours_on20,WB_Bin_20_22_Nhours_on20,WB_Bin_22_24_Nhours_on20,WB_Bin_24_26_Nhours_on20,WB_Bin_16_18_Nhours_on20,WB_Bin_14_16_Nhours_on20,WB_Bin_inf14_Nhours_on20,WB_Bin_26_28_Nhours_on20,WB_Bin_sup28_Nhours_on20,WB_Bin_18_20_Nhours_on10,WB_Bin_20_22_Nhours_on10,WB_Bin_22_24_Nhours_on10,WB_Bin_24_26_Nhours_on10,WB_Bin_16_18_Nhours_on10,WB_Bin_14_16_Nhours_on10,WB_Bin_inf14_Nhours_on10,WB_Bin_26_28_Nhours_on10,WB_Bin_sup28_Nhours_on10
,<int>,<dbl>,<dbl>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>
1,42022,74,15.25,183,153,180,91,82,25,4,2,0,110,112,119,55,54,25,4,1,0,53,35,52,24,47,25,4,0,0
2,42022,74,15.50,124,97,140,107,118,79,48,7,0,90,68,96,68,72,41,43,2,0,28,27,46,24,36,36,43,0,0


# Save

In [11]:
fwrite(result,"./output/df_WB_cells_CDC_Nhours.csv")